# Postgres DB Test

**First step:** Start Postgres in Docker (run in terminal):
```bash
cd /home/dev/projects/relative-r
docker compose up -d
```

Then run the cells below in order.

In [4]:
import sys
sys.path.insert(0, "..")

from src.db import make_engine, make_session_factory, health_check, Base
# Import models so they register with Base.metadata
from src.db.models import market, sector, industry, asset, pricerow

In [9]:
# 1. Create engine and test connection
engine = make_engine()
tables = health_check(engine)
print("Connected! Existing tables:", tables)

Connected! Existing tables: ['public.assets', 'public.industries', 'public.markets', 'public.pricerows', 'public.sectors']


In [6]:
# 2. Create all tables from models
Base.metadata.create_all(engine)
tables = health_check(engine)
print("Tables after create_all:", tables)

Tables after create_all: ['public.assets', 'public.industries', 'public.markets', 'public.pricerows', 'public.sectors']


In [7]:
# 3. Insert sample data
from src.db.models.market import Market as DbMarket
from src.db.models.sector import Sector as DbSector
from src.db.models.industry import Industry as DbIndustry

Session = make_session_factory(engine)
with Session() as session:
    m = DbMarket()
    session.add(m)
    session.flush()  # get market_id before commit

    s = DbSector()
    session.add(s)
    session.flush()

    i = DbIndustry(sector_id=s.sector_id)
    session.add(i)
    session.commit()
    print(f"Inserted: market_id={m.market_id}, sector_id={s.sector_id}, industry_id={i.industry_id}")

Inserted: market_id=1, sector_id=1, industry_id=1


In [8]:
# 4. Query and show
from sqlalchemy import text

with engine.connect() as conn:
    for table in ["markets", "sectors", "industries"]:
        result = conn.execute(text(f"SELECT * FROM {table}"))
        rows = result.fetchall()
        print(f"{table}:", rows)

markets: [(1, None)]
sectors: [(1,)]
industries: [(1, 1)]
